# 02 — Framework and Specification

**EPISTEMIC STATUS: EXPLANATORY** (with one clearly demarcated **ILLUSTRATIVE** section)

This notebook renders the five-gate architecture specification, risk-tiered thresholds, FUTURE-AI alignment mapping, shared governance service model, and glossary. It includes one carefully bounded illustrative worked example comparing compensatory and non-compensatory decision logic using synthetic data.

**Claim IDs (primary coverage in this notebook):** P1-C02 (illustrative contrast), P1-C04, P1-C05, P1-C13, P1-C15, P1-C18; companion-empirical citations P1-C06 / P1-C07 remain **EXTERNAL** (see `docs/claim_traceability.md`).

**STM targets:** STM-T2, STM-CF1, STM-CF2, STM-CF4, STM-CF5, STM-M1

**Outputs:** `outputs/tables/table2_rendered.csv`, `outputs/tables/gates_summary.csv`, `outputs/tables/futureai_alignment.csv`, `outputs/figures/illustrative_compensation_example.png`


## The Five Governance Gates

The gate architecture proposes five prerequisite governance domains. Each domain constitutes a checkpoint requiring a minimum evidence floor. The decision rule requires that all five pass for deployment to proceed (conjunctive / non-compensatory logic).

The five-domain decomposition reflects a convergence of requirements across reviewed regulatory instruments (EU AI Act, FDA SaMD Action Plan, MHRA guidance, ISO/IEC 42001, NICE Evidence Standards Framework, and FUTURE-AI) rather than an arbitrary partitioning.

**Alternative decompositions considered and rejected:**
- A four-gate model collapsing documentation and accountability was rejected because it would obscure whether deployment failure arose from absent documentation or from undefined withdrawal authority.
- A six-gate model separating calibration from broader safety validation was rejected because calibration is functionally inseparable from the clinical validity assessment.


In [1]:
import json
import hashlib
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Deterministic non-interactive backend
import matplotlib.pyplot as plt

# Lock matplotlib parameters for deterministic rendering
plt.rcParams.update({
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
})

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Load gate specification
with open(REPO_ROOT / 'data' / 'gates_specification.json', 'r', encoding='utf-8') as f:
    gates_data = json.load(f)

assert len(gates_data['gates']) == 5, "Expected 5 gates"

# Render gates summary table
gates_rows = []
for g in gates_data['gates']:
    gates_rows.append({
        'Gate': f"G{g['number']}",
        'Domain': g['domain'],
        'Overridable': 'Yes' if g['overridable'] else 'No (non-overridable)',
        'Evidence Floor (summary)': '; '.join(g['minimum_evidence_floor']),
    })

df_gates = pd.DataFrame(gates_rows)
output_gates = REPO_ROOT / 'outputs' / 'tables' / 'gates_summary.csv'
df_gates.to_csv(output_gates, index=False)
print(f"Written: {output_gates.relative_to(REPO_ROOT)}")
df_gates


Written: outputs\tables\gates_summary.csv


,Gate,Domain,Overridable,Evidence Floor (summary)
0,G1,Clinical Safety and Validation,No (non-overridable),External or temporal validation relevant to th...
1,G2,Bias and Equity Oversight,Yes,Subgroup performance evaluation across sociall...
2,G3,Documentation and Transparency,Yes,Model card or model facts label describing dat...
3,G4,Accountability and Decision Rights,Yes,Named accountable owner with explicit suspensi...
4,G5,Lifecycle Monitoring and Drift Management,No (non-overridable),"Predefined metrics covering performance, calib..."


## Gate-by-Gate Walkthrough

**Gate 1 — Clinical Safety and Validation (Non-Overridable):** Failure is non-overridable. A system without evidence of deployment-context performance cannot be safely activated. Requires external/temporal validation, calibration assessment, and a defined statement of intended use.

**Gate 2 — Bias and Equity Oversight (Overridable):** Addresses the risk that acceptable aggregate performance masks systematically worse outcomes for specific subpopulations. Override permitted where equity data collection is actively underway with a prospective monitoring plan.

**Gate 3 — Documentation and Transparency (Overridable):** Requires a model card or model facts label with data provenance and version control. Override permitted where an interim documentation record is accepted with a complete model card committed within a fixed period.

**Gate 4 — Accountability and Decision Rights (Overridable):** Requires a named accountable owner with suspension authority and separation between procurement incentives and deployment authorisation. Override permitted where an interim named lead is designated with board approval.

**Gate 5 — Lifecycle Monitoring and Drift Management (Non-Overridable):** Failure is non-overridable. A deployed system that cannot be monitored cannot be safely recalled. Requires predefined metrics and prespecified thresholds triggering review, pause, or withdrawal.


## Risk-Tiered Thresholds (Table 2)

To prevent evidence requirements from imposing disproportionate burdens on lower-risk applications, risk-tiered thresholds are proposed. The structure of the gates remains constant; the height of the minimum floor varies by risk class.


In [2]:
# Load and render Table 2
with open(REPO_ROOT / 'data' / 'table2_risk_tiers.json', 'r', encoding='utf-8') as f:
    t2_data = json.load(f)

assert len(t2_data['domains']) == 5, "Expected 5 domains"

t2_rows = []
for d in t2_data['domains']:
    t2_rows.append({
        'Domain': d['domain'],
        'Overridable': 'No' if not d['overridable'] else 'Yes',
        'Standard Risk (Minimum Floor)': d['standard_risk_floor'],
        'Higher Risk (Enhanced Floor)': d['higher_risk_floor'],
    })

df_t2 = pd.DataFrame(t2_rows)
output_t2 = REPO_ROOT / 'outputs' / 'tables' / 'table2_rendered.csv'
df_t2.to_csv(output_t2, index=False)
print(f"Written: {output_t2.relative_to(REPO_ROOT)}")
df_t2


Written: outputs\tables\table2_rendered.csv


,Domain,Overridable,Standard Risk (Minimum Floor),Higher Risk (Enhanced Floor)
0,Safety (G1),No,Intended-use statement; representative interna...,Prospective external validation; CONSORT-AI al...
1,Equity (G2),Yes,Subgroup reporting across protected characteri...,Intersectional analysis; documented mitigation...
2,Documentation (G3),Yes,Model card with data provenance and exclusion ...,Full technical file; version-controlled tracea...
3,Accountability (G4),Yes,Named withdrawal authority; escalation pathway.,Board sign-off; AI Clinical Lead with protecte...
4,Monitoring (G5),No,Monthly performance audits with documented rev...,Real-time drift detection; automated alert thr...


## FUTURE-AI Compatibility Mapping

The gate architecture is compatible with and complementary to the FUTURE-AI consensus guideline. The following mapping shows how FUTURE-AI's six principles align with the five governance gate domains.

> **Note:** This is the author's interpretive mapping. The gate architecture does not compete with FUTURE-AI; it provides the decision-theoretic layer that converts FUTURE-AI evidence into an institutional deployment decision.


In [3]:
# Load and render FUTURE-AI mapping
with open(REPO_ROOT / 'data' / 'futureai_mapping.json', 'r', encoding='utf-8') as f:
    fai_data = json.load(f)

fai_rows = [{'FUTURE-AI Principle': m['futureai_principle'], 'Gate Domain': m['gate_domain'], 'Alignment': m['alignment_note']} for m in fai_data['mappings']]
df_fai = pd.DataFrame(fai_rows)
output_fai = REPO_ROOT / 'outputs' / 'tables' / 'futureai_alignment.csv'
df_fai.to_csv(output_fai, index=False)
print(f"Written: {output_fai.relative_to(REPO_ROOT)}")
df_fai


Written: outputs\tables\futureai_alignment.csv


,FUTURE-AI Principle,Gate Domain,Alignment
0,Fairness,Gate 2 (Bias and Equity Oversight),Fairness aligns with equity oversight requirem...
1,Universality,Gates 1-5 (cross-cutting),Universality spans multiple gate domains as a ...
2,Traceability,Gate 3 (Documentation and Transparency),Traceability aligns with documentation require...
3,Usability,Gate 1 (Safety) and Gate 4 (Accountability),Usability aligns with safety validation and ac...
4,Robustness,Gate 5 (Lifecycle Monitoring and Drift Managem...,Robustness aligns with monitoring requirements
5,Explainability,Gate 3 (Documentation and Transparency),Explainability aligns with documentation and t...


## Shared Governance Service Model

Resource-constrained institutions may implement gate governance through shared service arrangements:

1. **Pooled audit capacity** across institutions for independent Gate 1 safety validation
2. **Governance-as-a-service** for documentation and accountability infrastructure (Gates 3–4)
3. **Consortium-based evidence pooling** for equity and monitoring evidence

The deploying institution retains ultimate liability regardless of external service provision.


## Glossary / Abbreviations


In [4]:
# Load and render glossary
with open(REPO_ROOT / 'data' / 'glossary.json', 'r', encoding='utf-8') as f:
    glos_data = json.load(f)

df_glos = pd.DataFrame(glos_data['terms'])
df_glos.columns = ['Abbreviation', 'Expansion']
df_glos


,Abbreviation,Expansion
0,AI,artificial intelligence
1,CONSORT-AI,Consolidated Standards of Reporting Trials for...
2,DTAC,Digital Technology Assessment Criteria
3,EU,European Union
4,FDA,Food and Drug Administration
5,MCDA,multi-criteria decision analysis
6,MHRA,Medicines and Healthcare products Regulatory A...
7,NHS,National Health Service
8,NICE,National Institute for Health and Care Excellence
9,SaMD,software as a medical device


---

## ⚠️ ILLUSTRATIVE SECTION BOUNDARY

Everything below this line uses **synthetic data for pedagogical purposes only**. It does not reproduce any finding from the manuscript or companion studies. The numbers are entirely invented to demonstrate how compensatory and non-compensatory decision rules differ.

---


In [5]:
# ILLUSTRATIVE EXAMPLE: Compensatory vs Non-Compensatory Decision Logic
#
# Scenario: A hypothetical AI system evaluated across 5 governance domains.
# Four domains score well. One domain (Safety) scores critically low.
#
# ALL NUMBERS ARE SYNTHETIC AND INVENTED FOR PEDAGOGICAL PURPOSES.

domains = ['Safety\n(G1)', 'Equity\n(G2)', 'Documentation\n(G3)', 'Accountability\n(G4)', 'Monitoring\n(G5)']
scores = [0.30, 0.85, 0.90, 0.88, 0.82]
weights = [0.20, 0.20, 0.20, 0.20, 0.20]  # Equal weights
minimum_floor = 0.50

# Compensatory: weighted average
composite_score = sum(s * w for s, w in zip(scores, weights))
compensatory_threshold = 0.60
compensatory_passes = composite_score >= compensatory_threshold

# Non-compensatory: all domains must meet minimum floor
domain_passes = [s >= minimum_floor for s in scores]
conjunctive_passes = all(domain_passes)

print("=== ILLUSTRATIVE EXAMPLE (Synthetic Data) ===")
print(f"Domain scores: {dict(zip(['G1','G2','G3','G4','G5'], scores))}")
print(f"Minimum floor: {minimum_floor}")
print(f"Compensatory composite score: {composite_score:.2f} (threshold: {compensatory_threshold}) -> {'PASS' if compensatory_passes else 'FAIL'}")
print(f"Conjunctive gate check: {dict(zip(['G1','G2','G3','G4','G5'], domain_passes))} -> {'PASS' if conjunctive_passes else 'FAIL'}")
print(f"\nKey insight: Safety scores {scores[0]} (below {minimum_floor} floor)")
print(f"  Compensatory scoring: MASKS the safety gap (composite {composite_score:.2f} > {compensatory_threshold})")
print(f"  Conjunctive gates: BLOCKS deployment (Safety fails minimum floor)")


=== ILLUSTRATIVE EXAMPLE (Synthetic Data) ===
Domain scores: {'G1': 0.3, 'G2': 0.85, 'G3': 0.9, 'G4': 0.88, 'G5': 0.82}
Minimum floor: 0.5
Compensatory composite score: 0.75 (threshold: 0.6) -> PASS
Conjunctive gate check: {'G1': False, 'G2': True, 'G3': True, 'G4': True, 'G5': True} -> FAIL

Key insight: Safety scores 0.3 (below 0.5 floor)
  Compensatory scoring: MASKS the safety gap (composite 0.75 > 0.6)
  Conjunctive gates: BLOCKS deployment (Safety fails minimum floor)


In [6]:
# Generate comparison figure
fig, ax = plt.subplots(figsize=(8, 5))

x = range(len(domains))
colors = ['#d32f2f' if not p else '#388e3c' for p in domain_passes]

bars = ax.bar(x, scores, color=colors, width=0.6, edgecolor='white', linewidth=0.5)

# Floor line
ax.axhline(y=minimum_floor, color='#1565c0', linestyle='--', linewidth=1.5, label=f'Minimum floor ({minimum_floor})')

# Composite score line
ax.axhline(y=composite_score, color='#ff8f00', linestyle='-.', linewidth=1.5, label=f'Composite score ({composite_score:.2f})')

# Compensatory threshold line
ax.axhline(y=compensatory_threshold, color='#ff8f00', linestyle=':', linewidth=1.0, alpha=0.7, label=f'Compensatory threshold ({compensatory_threshold})')

ax.set_xticks(x)
ax.set_xticklabels(domains, fontsize=9)
ax.set_ylabel('Score', fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title('Illustrative Example: Compensatory vs Non-Compensatory Logic\n(Synthetic data — not an empirical finding)', fontsize=11, style='italic')
ax.legend(loc='lower right', fontsize=8)

# Annotations
ax.annotate('BLOCKED\nby gate', xy=(0, scores[0]), xytext=(0.5, 0.15),
            fontsize=8, color='#d32f2f', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#d32f2f', lw=1.0))

ax.annotate('Composite PASSES\n(masks safety gap)', xy=(2, composite_score), xytext=(2.8, composite_score + 0.12),
            fontsize=8, color='#ff8f00', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#ff8f00', lw=1.0))

fig.tight_layout()
output_fig = REPO_ROOT / 'outputs' / 'figures' / 'illustrative_compensation_example.png'
fig.savefig(output_fig, dpi=150)
plt.close(fig)
print(f"Written: {output_fig.relative_to(REPO_ROOT)}")


Written: outputs\figures\illustrative_compensation_example.png


### What This Shows

Under compensatory scoring, the weighted aggregate (0.75) exceeds the composite threshold (0.60), allowing deployment despite a critical safety gap (0.30). Under conjunctive gates, the safety domain's failure to meet the minimum floor (0.50) blocks deployment regardless of other domain strengths.

This is the core decision-theoretic insight of the Viewpoint: in a conjunctive architecture, no domain can compensate for failure in another.

### What This Does NOT Show

This is **not empirical evidence**. The numbers are synthetic and invented. The companion simulation study [25] provides empirical characterisation of the conditions under which the gate safety advantage holds or disappears, including the finding that the advantage diminishes to zero when unsafe tools fail uniformly across all domains.


In [7]:
# Output hashes for manifest validation
for fpath in [output_gates, output_t2, output_fai, output_fig]:
    h = hashlib.sha256()
    with open(fpath, 'rb') as f2:
        h.update(f2.read())
    print(f"SHA-256 of {fpath.name}: {h.hexdigest()}")


SHA-256 of gates_summary.csv: 467cfb3dc1f9b32fdb4d4a305a59e15abfc3386ec220da89c58bd0b19a5714f8
SHA-256 of table2_rendered.csv: b2de656f3bb98b46b72c0d504bc7693ba45e1b10eaccdb607f2d93559b7514e7
SHA-256 of futureai_alignment.csv: 5a9b479727e448879c4014f1fb404242d0dbaab2474c66347e1801a848a96c14
SHA-256 of illustrative_compensation_example.png: 0c9412a7891c888ff0553bcfad42825b4efb7a8aabbd537a206f539e526f2c11
